# PSX Compliance Agent — Qwen (DashScope)

Run cells top to bottom. To start a **fresh conversation** (new memory), re-run the *thread id* cell and then the *context* cell.

To ask a new question: edit the *user input* cell, then run the *invoke* cell. Ensure your `.env` has `DASHSCOPE_API_KEY`, and select the `.venv` kernel.

In [ ]:
import os
import uuid
from dotenv import load_dotenv

load_dotenv(override=True)  # reads .env; override=True so .env wins over any stale system env vars
os.environ.setdefault("DB_PATH", os.path.join(os.getcwd(), "data", "psx.db"))

from app.common.utils import _get_model
from app.common.context import SessionContext
from app.graph.compile import get_compiled_agent

assert os.getenv("DASHSCOPE_API_KEY"), "Set DASHSCOPE_API_KEY in .env (see .env.example)"
print("DB_PATH =", os.environ["DB_PATH"])
print("model   =", os.getenv("QWEN_MODEL", "qwen-plus"))

DB_PATH = data/psx.db
model   = qwen-flash


In [ ]:
# --- LangSmith tracing (EU region) ---
# Your key is an EU-region key, so use the EU endpoint. Add LANGSMITH_API_KEY to .env.
# Use the canonical LANGSMITH_* names only (do NOT also set LANGCHAIN_ENDPOINT -- mixing
# the two causes endpoint conflicts). Run this BEFORE the first invoke in a fresh kernel.
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"
os.environ["LANGSMITH_PROJECT"] = "devpost-hackathon"

assert os.getenv("LANGSMITH_API_KEY"), "Add LANGSMITH_API_KEY (EU region) to .env"
print("LangSmith tracing ON -> project:", os.environ["LANGSMITH_PROJECT"], "| endpoint: EU")

In [9]:
# Compile the agent once.
agent = await get_compiled_agent("compliance_agent")

In [10]:
# Thread id -- re-run this cell to start a brand-new conversation (fresh memory).
thread_id = str(uuid.uuid4())
print("thread_id =", thread_id)

thread_id = d7e703bb-78d9-49cb-bf4d-ae9dd82d00e6


In [11]:
# Context -- edit freely (e.g. swap model tier). Re-run after changing thread_id above.
context = SessionContext(
    thread_id=thread_id,
    model=_get_model("qwen"),
)
config = {"configurable": {"thread_id": thread_id}}

In [12]:
# User input -- edit this, then run the invoke cell below.
user_input = "Which broker has the highest total fines, and how much (in PKR)?"

In [ ]:
# Invoke the agent with the user input + context, streaming the answer.
async for chunk in agent.astream(
    {"messages": [{"role": "user", "content": user_input}]},
    stream_mode="custom",
    context=context,
    config=config,
):
    if chunk and "compliance_chunk" in chunk:
        print(chunk["compliance_chunk"], end="", flush=True)
print()

The broker with the highest total fines is **New Peak Securities (Private) Limited**, with a total of **PKR 8,656,646** in fines.


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://eu.api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://eu.api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://eu.api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://eu.api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
